# 13 - Eficiencia en GPUs

**AI sin humo** - Notas personales para entender deep learning desde cero.

Entrenar modelos grandes como GPT requiere GPUs. Pero usar GPUs eficientemente no es trivial: hay que entender su arquitectura, cómo aprovechar su memoria y compute, y qué técnicas usar para maximizar el throughput.

Este notebook cubre:
- Arquitectura de GPUs (SMs, tensor cores, memoria)
- Recursos limitantes (memoria, bandwidth, FLOPs)
- Escalado de memoria en modelos grandes
- Compute-bound vs memory-bound
- Técnicas de optimización (Flash Attention, mixed precision, etc.)

---

## Contenido

1. [Arquitectura de GPUs](#arquitectura)
2. [Jerarquía de memoria](#memoria)
3. [Recursos: Memory, Bandwidth, FLOPs](#recursos)
4. [Escalado de memoria en LLMs](#escalado)
5. [Compute-bound vs Memory-bound](#bound)
6. [Attention es cuadrático en T](#attention)
7. [Técnicas de optimización](#optimizacion)
8. [Resumen](#resumen)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import math
import time

# Check GPU availability
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("No GPU available")

---

<a id='arquitectura'></a>
## 1. Arquitectura de GPUs

![GPU Architecture](../ai_notas/AI%20notas/image%2061.png)

Las GPUs modernas (NVIDIA) tienen una arquitectura jerárquica:

### Streaming Multiprocessors (SMs)

Un SM es una unidad de procesamiento que contiene:
- **CUDA Cores**: procesadores escalares que ejecutan operaciones básicas (add, multiply)
- **Tensor Cores**: unidades especializadas para operaciones de matriz (GEMM: General Matrix Multiply)
  - Mucho más rápidos que CUDA cores para matrices
  - Operan en bloques de 16x16 (FP16) o 8x8 (FP32)
  - Clave para transformers (muchas multiplicaciones de matrices)

### Ejemplo: A100 GPU

- **108 SMs** (cada uno con 64 CUDA cores + 4 tensor cores)
- Total: ~7000 CUDA cores, 432 tensor cores
- Memoria HBM: 40GB o 80GB
- Bandwidth: ~2 TB/s

### Paralelismo masivo

Las GPUs ejecutan **miles de threads en paralelo**. Cada thread puede ejecutar operaciones independientes. Esto es perfecto para deep learning donde:
- Operaciones sobre batches (cada ejemplo independiente)
- Operaciones sobre tokens (cada posición independiente)
- Operaciones sobre canales (cada feature map independiente)

---

<a id='memoria'></a>
## 2. Jerarquía de memoria

![Memory Hierarchy](../ai_notas/AI%20notas/image%2062.png)

La memoria en GPUs tiene múltiples niveles, cada uno con diferente tamaño y velocidad:

### HBM (High Bandwidth Memory)

- **Más grande** pero **más lenta** (relativamente)
- Típicamente 40-80 GB en GPUs modernas
- Bandwidth: ~2 TB/s (A100)
- Es donde viven los parámetros del modelo, activaciones, y datos

### L2 Cache

- Cache compartido por todos los SMs
- Típicamente 40-50 MB
- Más rápido que HBM, más lento que L1
- Reduce accesos a HBM

### L1 Cache / Shared Memory

- Cache por SM (típicamente 128-164 KB por SM)
- Muy rápido
- Compartido entre threads del mismo SM
- Útil para operaciones que requieren comunicación entre threads

### Registros

- El nivel más rápido
- Privados por thread
- Limitados (~65K registros por SM, repartidos entre threads)

### Implicaciones

Para ser eficiente, hay que:
1. **Minimizar accesos a HBM**: mantener datos en cache cuando sea posible
2. **Reutilizar datos**: si lees algo de HBM, úsalo muchas veces antes de descartarlo
3. **Coalescing**: acceder a memoria de forma contigua (los threads vecinos leen posiciones vecinas)

In [ ]:
# Visualize memory hierarchy speeds
memory_levels = ['Registers', 'L1/Shared', 'L2 Cache', 'HBM']
latency_ns = [1, 20, 200, 1000]  # approximate latencies
size_gb = [0.0001, 0.0002, 0.05, 40]  # approximate sizes

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Latency
ax1.barh(memory_levels, latency_ns, color=['green', 'lightgreen', 'orange', 'red'])
ax1.set_xlabel('Latency (ns, log scale)')
ax1.set_xscale('log')
ax1.set_title('Memory Latency')
ax1.grid(axis='x', alpha=0.3)

# Size
ax2.barh(memory_levels, size_gb, color=['green', 'lightgreen', 'orange', 'red'])
ax2.set_xlabel('Size (GB, log scale)')
ax2.set_xscale('log')
ax2.set_title('Memory Size')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("Trade-off: más rápido = más pequeño, más lento = más grande")

---

<a id='recursos'></a>
## 3. Recursos: Memory, Bandwidth, FLOPs

Hay tres recursos principales que limitan el entrenamiento:

### 1. Memoria (Memory)

**¿Cuánto cabe?** Determina el tamaño máximo del modelo y batch size.

Ejemplo A100:
- 40GB o 80GB HBM
- Debe contener: parámetros del modelo, optimizer states, activaciones, gradientes

### 2. Bandwidth (Ancho de banda)

**¿Qué tan rápido puedo leer/escribir?** Determina cuánto tiempo se pierde moviendo datos.

Ejemplo A100:
- ~2 TB/s de bandwidth HBM
- Si necesitas leer 1GB, toma ~0.5ms
- Si haces muchas operaciones pequeñas que requieren leer de HBM, el bandwidth es el cuello de botella

### 3. FLOPs (Floating Point Operations)

**¿Qué tan rápido puedo calcular?** Determina cuántas operaciones matemáticas por segundo.

Ejemplo A100:
- ~312 TFLOPS (FP16 con tensor cores)
- ~19.5 TFLOPS (FP32 con CUDA cores)
- Los tensor cores son ~16× más rápidos para operaciones de matriz

### El recurso limitante

Dependiendo de la operación, uno de estos será el cuello de botella:
- **Memory-bound**: operaciones simples que leen mucho pero calculan poco (ej: activación ReLU)
- **Compute-bound**: operaciones complejas que calculan mucho (ej: multiplicación de matrices grandes)

In [ ]:
# Example: Memory vs Compute bound operations

def memory_bound_operation(x):
    """Simple operation: reads a lot, computes little."""
    return torch.relu(x)  # Just thresholding, very fast compute

def compute_bound_operation(x):
    """Complex operation: computes a lot."""
    return x @ x.T  # Matrix multiplication, lots of FLOPs

# Simulate
size = 10000
x = torch.randn(size, size)

if torch.cuda.is_available():
    x = x.cuda()
    
    # Memory-bound: bandwidth limited
    torch.cuda.synchronize()
    start = time.time()
    _ = memory_bound_operation(x)
    torch.cuda.synchronize()
    mem_time = time.time() - start
    
    # Compute-bound: FLOPs limited
    torch.cuda.synchronize()
    start = time.time()
    _ = compute_bound_operation(x)
    torch.cuda.synchronize()
    compute_time = time.time() - start
    
    print(f"Memory-bound (ReLU): {mem_time*1000:.2f} ms")
    print(f"Compute-bound (MatMul): {compute_time*1000:.2f} ms")
    print(f"\nMatMul es {compute_time/mem_time:.1f}x más lento (más FLOPs)")
else:
    print("No GPU available for timing")

---

<a id='escalado'></a>
## 4. Escalado de memoria en LLMs

Entrenar un LLM requiere memoria para:

### 1. Parámetros del modelo

Para un transformer con `L` capas, `d` dimensiones, `f` factor de expansión del FFN:

**Attention por capa:**
- Q, K, V projections: `3 × d²` parámetros
- Output projection: `d²` parámetros
- Total attention: `4d²` parámetros por capa

**MLP (Feed-Forward) por capa:**
- Up projection: `d × (f×d)` = `fd²`
- Down projection: `(f×d) × d` = `fd²`
- Total MLP: `2fd²` parámetros por capa (típicamente f=4, entonces `8d²`)

**Por capa:** `4d² + 2fd²` ≈ `12d²` (con f=4)

**Total modelo:** `L × 12d²` parámetros

Ejemplo GPT-3 (175B):
- L=96, d=12288 → ~175B parámetros
- En FP32: 175B × 4 bytes = **700 GB** (¡demasiado!)
- En FP16: 175B × 2 bytes = **350 GB** (todavía mucho)

### 2. Optimizer states

**Adam optimizer** guarda:
- `momentum` (m): mismo tamaño que parámetros
- `variance` (v): mismo tamaño que parámetros
- Total: **2×** el tamaño de parámetros

**AdamW con FP32 states:**
- Parámetros en FP16: `P` bytes
- Optimizer states en FP32: `2P × 2` = `4P` bytes (2× por ser FP32)
- **Total: 5P bytes** (5× los parámetros)

### 3. Activaciones

Durante el forward pass, se guardan activaciones para el backward pass:

**Por token en una capa:**
- Input embeddings: `d` floats
- Attention output: `d` floats
- FFN intermediate: `fd` floats
- Total por token: `(2 + f)d` ≈ `6d` (con f=4)

**Por batch:**
- `B × T × 6d × L` floats
- En FP16: `B × T × 6d × L × 2` bytes

Ejemplo GPT-3 con batch=1, seq_len=2048:
- `1 × 2048 × 6 × 12288 × 96 × 2` = **~36 GB** solo activaciones

### Resumen de memoria

Para entrenar un modelo con `P` parámetros:
- Parámetros (FP16): `P × 2` bytes
- Optimizer states (FP32): `P × 8` bytes
- Activaciones: depende de batch size y seq_len, pero típicamente `~0.5-2×` parámetros

**Total aproximado:** `~10-12×` el tamaño de parámetros en bytes

In [ ]:
# Calculate memory requirements for different model sizes

def calculate_memory_requirements(d, L, f=4, B=1, T=2048, fp16_params=True):
    """Calculate memory requirements for a transformer model."""
    # Parameters per layer
    params_per_layer = 4 * d * d + 2 * f * d * d  # attention + MLP
    total_params = L * params_per_layer
    
    # Memory breakdown
    if fp16_params:
        param_memory = total_params * 2  # FP16: 2 bytes
        optimizer_memory = total_params * 8  # FP32 states: 4 bytes × 2 states
    else:
        param_memory = total_params * 4  # FP32: 4 bytes
        optimizer_memory = total_params * 8  # FP32 states
    
    # Activations (approximate)
    activations_per_token = (2 + f) * d  # attention + FFN intermediate
    activation_memory = B * T * activations_per_token * L * 2  # FP16
    
    total_memory = param_memory + optimizer_memory + activation_memory
    
    return {
        'params': total_params,
        'param_memory_gb': param_memory / 1e9,
        'optimizer_memory_gb': optimizer_memory / 1e9,
        'activation_memory_gb': activation_memory / 1e9,
        'total_memory_gb': total_memory / 1e9
    }

# Examples
models = [
    {'name': 'GPT-2 Small', 'd': 768, 'L': 12},
    {'name': 'GPT-2 Medium', 'd': 1024, 'L': 24},
    {'name': 'GPT-3 6.7B', 'd': 4096, 'L': 32},
    {'name': 'GPT-3 175B', 'd': 12288, 'L': 96},
]

print("Memory Requirements (FP16 params, FP32 optimizer states):")
print("=" * 80)
for model in models:
    mem = calculate_memory_requirements(model['d'], model['L'])
    print(f"\n{model['name']}:")
    print(f"  Params: {mem['params']/1e9:.2f}B")
    print(f"  Param memory: {mem['param_memory_gb']:.2f} GB")
    print(f"  Optimizer memory: {mem['optimizer_memory_gb']:.2f} GB")
    print(f"  Activation memory: {mem['activation_memory_gb']:.2f} GB")
    print(f"  Total: {mem['total_memory_gb']:.2f} GB")

---

<a id='bound'></a>
## 5. Compute-bound vs Memory-bound

Una operación es **compute-bound** o **memory-bound** dependiendo de su **arithmetic intensity**.

### Arithmetic Intensity

$$\text{Arithmetic Intensity} = \frac{\text{FLOPs}}{\text{Bytes leídos/escritos}}$$

- **Alta intensidad** (>100 FLOPs/byte): compute-bound
  - Ejemplo: multiplicación de matrices grandes (muchos FLOPs, pocos accesos a memoria)
  - Limitado por FLOPs disponibles
  
- **Baja intensidad** (<10 FLOPs/byte): memory-bound
  - Ejemplo: activación ReLU (pocos FLOPs, muchos accesos a memoria)
  - Limitado por bandwidth de memoria

### Ejemplo: Matrix Multiplication

Para `C = A @ B` donde A es `(M, K)` y B es `(K, N)`:

- **FLOPs**: `M × K × N` multiplicaciones + `M × K × N` sumas = `2MNK` FLOPs
- **Bytes**: leer A (`M×K×4`), leer B (`K×N×4`), escribir C (`M×N×4`) = `4(MK + KN + MN)` bytes
- **Intensity**: `2MNK / 4(MK + KN + MN)`

Si M=N=K=4096:
- Intensity ≈ `2×4096³ / 4×3×4096²` ≈ **682 FLOPs/byte** → **compute-bound**

### Ejemplo: Element-wise Operation

Para `C = relu(A)` donde A es `(M, N)`:

- **FLOPs**: `M×N` comparaciones ≈ `MN` FLOPs
- **Bytes**: leer A (`M×N×4`), escribir C (`M×N×4`) = `8MN` bytes
- **Intensity**: `MN / 8MN` = **0.125 FLOPs/byte** → **memory-bound**

### Implicaciones

- **Compute-bound**: optimizar usando tensor cores, batch sizes grandes, operaciones fusionadas
- **Memory-bound**: optimizar accesos a memoria (coalescing, cache, reducir accesos redundantes)

In [ ]:
# Visualize arithmetic intensity for different operations

def calculate_intensity(flops, bytes_read_write):
    return flops / bytes_read_write if bytes_read_write > 0 else 0

operations = [
    {'name': 'ReLU (element-wise)', 'flops': 1e6, 'bytes': 8e6, 'color': 'red'},
    {'name': 'LayerNorm', 'flops': 2e6, 'bytes': 12e6, 'color': 'orange'},
    {'name': 'MatMul 1K×1K', 'flops': 2e9, 'bytes': 12e6, 'color': 'green'},
    {'name': 'MatMul 4K×4K', 'flops': 2*4096**3, 'bytes': 4*3*4096**2, 'color': 'darkgreen'},
]

intensities = [calculate_intensity(op['flops'], op['bytes']) for op in operations]
names = [op['name'] for op in operations]
colors = [op['color'] for op in operations]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(names, intensities, color=colors)
ax.set_xlabel('Arithmetic Intensity (FLOPs/byte)', fontsize=12)
ax.set_xscale('log')
ax.set_title('Compute-bound vs Memory-bound Operations', fontsize=14)
ax.axvline(x=100, color='gray', linestyle='--', label='High intensity threshold')
ax.axvline(x=10, color='gray', linestyle=':', label='Low intensity threshold')
ax.legend()
ax.grid(axis='x', alpha=0.3)

# Add labels
for i, (bar, intensity) in enumerate(zip(bars, intensities)):
    ax.text(intensity * 1.1, i, f'{intensity:.1f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\nOperations above ~100 FLOPs/byte are compute-bound (limited by FLOPs)")
print("Operations below ~10 FLOPs/byte are memory-bound (limited by bandwidth)")

---

<a id='attention'></a>
## 6. Attention es cuadrático en T

El problema más grande de los transformers: **attention escala cuadráticamente** con la longitud de secuencia.

### Complejidad de Attention

Para una secuencia de longitud `T`:

1. **Compute**: `Q @ K^T` produce matriz `(T, T)` → `O(T²)` operaciones
2. **Memory**: guardar matriz `(T, T)` → `O(T²)` memoria

### Ejemplo práctico

- `T = 512`: matriz de atención `512×512` = 1M elementos
- `T = 2048`: matriz de atención `2048×2048` = 4M elementos (4× más)
- `T = 8192`: matriz de atención `8192×8192` = 67M elementos (64× más que 512)

### Impacto en memoria

Con batch size `B` y `H` heads:
- Memoria para scores: `B × H × T × T × 4` bytes (FP32)
- Con `B=1, H=12, T=8192`: `1 × 12 × 8192² × 4` = **3.2 GB** solo para scores

### Por qué es un problema

- Secuencias largas son esenciales para contexto (código, documentos largos)
- Pero el costo cuadrático limita qué tan largo puede ser el contexto
- Muchas técnicas de optimización (Flash Attention, etc.) intentan reducir este costo

In [ ]:
# Visualize quadratic scaling of attention

T_values = np.array([512, 1024, 2048, 4096, 8192, 16384])
memory_mb = (T_values ** 2) * 4 / 1e6  # FP32, single head, single batch
compute_flops = (T_values ** 2) * 512  # approximate FLOPs

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Memory scaling
ax1.plot(T_values, memory_mb, 'o-', linewidth=2, markersize=8)
ax1.set_xlabel('Sequence Length (T)', fontsize=12)
ax1.set_ylabel('Memory per Attention Matrix (MB)', fontsize=12)
ax1.set_title('Memory: O(T²)', fontsize=14)
ax1.grid(alpha=0.3)
ax1.set_xscale('log')
ax1.set_yscale('log')

# Compute scaling
ax2.plot(T_values, compute_flops / 1e9, 'o-', linewidth=2, markersize=8, color='orange')
ax2.set_xlabel('Sequence Length (T)', fontsize=12)
ax2.set_ylabel('FLOPs (GFLOPs)', fontsize=12)
ax2.set_title('Compute: O(T²)', fontsize=14)
ax2.grid(alpha=0.3)
ax2.set_xscale('log')
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

print("Attention memory and compute both scale quadratically with sequence length")
print("This is why long contexts are expensive!")

---

<a id='optimizacion'></a>
## 7. Técnicas de optimización

### 1. Data Loading

**Problema**: el GPU puede procesar datos mucho más rápido de lo que el CPU puede cargarlos del disco.

**Solución**:
- **Multiple workers**: cargar datos en paralelo
- **Prefetching**: cargar el siguiente batch mientras se procesa el actual
- **DataLoader con pin_memory**: transferencia más rápida CPU→GPU

```python
dataloader = DataLoader(
    dataset,
    batch_size=32,
    num_workers=4,      # paralelismo
    pin_memory=True,    # faster CPU→GPU transfer
    prefetch_factor=2  # prefetch batches
)
```

### 2. torch.compile (PyTorch 2.0+)

**Problema**: PyTorch ejecuta operaciones una por una, con overhead de Python.

**Solución**: `torch.compile` compila el modelo a código optimizado (similar a JAX).

```python
model = torch.compile(model)  # One line!
```

Beneficios:
- Fusiona operaciones (reduce kernel launches)
- Optimiza accesos a memoria
- Mejoras típicas: 1.5-2× más rápido

### 3. Flash Attention

**Problema**: attention estándar materializa la matriz `(T, T)` completa en memoria.

**Solución**: Flash Attention calcula attention en bloques, sin materializar la matriz completa.

- **Memoria**: `O(T)` en lugar de `O(T²)`
- **Velocidad**: más rápido también (mejor uso de cache)
- Implementación: `flash_attn` library

```python
from flash_attn import flash_attn_func

# Instead of standard attention
output = flash_attn_func(q, k, v, dropout_p=0.0, softmax_scale=1.0)
```

### 4. Mixed Precision (AMP)

**Problema**: FP32 es lento y ocupa mucho espacio.

**Solución**: usar FP16 para forward/backward, FP32 solo para optimizer states.

- **Velocidad**: 2× más rápido (tensor cores usan FP16)
- **Memoria**: 2× menos espacio para activaciones
- **Estabilidad**: mantiene precisión con loss scaling

```python
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()

with autocast():
    outputs = model(inputs)
    loss = criterion(outputs, targets)

scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()
```

### 5. Gradient Accumulation

**Problema**: batch size grande no cabe en memoria.

**Solución**: acumular gradientes sobre múltiples micro-batches antes de hacer `optimizer.step()`.

- Efectivamente aumenta el batch size sin aumentar memoria
- Trade-off: más tiempo de entrenamiento (más forward/backward passes)

```python
accumulation_steps = 4
optimizer.zero_grad()

for i, batch in enumerate(dataloader):
    loss = model(batch) / accumulation_steps
    loss.backward()
    
    if (i + 1) % accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad()
```

### 6. Distributed Data Parallel (DDP)

**Problema**: una GPU no es suficiente para modelos grandes.

**Solución**: entrenar en múltiples GPUs en paralelo.

- Cada GPU tiene una copia del modelo
- Cada GPU procesa un batch diferente
- Gradientes se sincronizan entre GPUs (promedio)
- Escalado casi lineal con número de GPUs

```python
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

# Initialize
dist.init_process_group("nccl")
model = DDP(model, device_ids=[rank])

# Training loop (same as single GPU)
# Gradients automatically synchronized
```

### Resumen de mejoras típicas

| Técnica | Speedup | Memory Reduction |
|---------|---------|------------------|
| torch.compile | 1.5-2× | - |
| Flash Attention | 2-3× | 10-100× (depende de T) |
| Mixed Precision | 2× | 2× |
| Gradient Accumulation | - | Permite batch size efectivo mayor |
| DDP | ~N× (N GPUs) | - |

In [ ]:
# Example: Mixed Precision training

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(1000, 2000),
            nn.ReLU(),
            nn.Linear(2000, 1000),
        )
    
    def forward(self, x):
        return self.layers(x)

# Setup
model = SimpleModel()
if torch.cuda.is_available():
    model = model.cuda()
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters())
    
    # Mixed precision setup
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler()
    
    # Dummy data
    x = torch.randn(32, 1000).cuda()
    y = torch.randn(32, 1000).cuda()
    
    # Training step with mixed precision
    optimizer.zero_grad()
    
    with autocast():  # FP16 forward
        outputs = model(x)
        loss = criterion(outputs, y)
    
    scaler.scale(loss).backward()  # FP16 backward with scaling
    scaler.step(optimizer)  # FP32 optimizer step
    scaler.update()  # Update scale factor
    
    print("Mixed precision training step completed!")
    print(f"Loss: {loss.item():.4f}")
else:
    print("No GPU available for mixed precision example")

In [ ]:
# Example: Gradient Accumulation

def train_with_accumulation(model, dataloader, optimizer, accumulation_steps=4):
    """Train with gradient accumulation to simulate larger batch size."""
    model.train()
    optimizer.zero_grad()
    
    total_loss = 0
    for i, (inputs, targets) in enumerate(dataloader):
        if torch.cuda.is_available():
            inputs, targets = inputs.cuda(), targets.cuda()
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # Scale loss by accumulation steps
        loss = loss / accumulation_steps
        
        # Backward pass (accumulates gradients)
        loss.backward()
        
        total_loss += loss.item()
        
        # Update weights every accumulation_steps
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            print(f"Step {i+1}: Loss = {total_loss:.4f}")
            total_loss = 0

print("Gradient accumulation allows effective batch size = batch_size × accumulation_steps")
print("Without increasing memory usage!")

---

<a id='resumen'></a>
## 8. Resumen

| Concepto | Descripción |
|----------|-------------|
| **SMs (Streaming Multiprocessors)** | Unidades de procesamiento con CUDA cores y tensor cores. Paralelismo masivo. |
| **Tensor Cores** | Unidades especializadas para multiplicación de matrices. ~16× más rápidas que CUDA cores para GEMM. |
| **Jerarquía de memoria** | HBM (grande, lento) → L2 → L1/Shared → Registros (pequeño, rápido). Minimizar accesos a HBM. |
| **Memory/Bandwidth/FLOPs** | Tres recursos limitantes. Memory: qué cabe. Bandwidth: velocidad de lectura/escritura. FLOPs: velocidad de cálculo. |
| **Escalado de memoria** | Parámetros: `L × 12d²`. Optimizer: `2×` parámetros (FP32). Activaciones: `B × T × 6d × L`. Total: ~10-12× parámetros. |
| **Compute-bound vs Memory-bound** | Depende de arithmetic intensity (FLOPs/byte). Alta intensidad (>100): compute-bound. Baja intensidad (<10): memory-bound. |
| **Attention cuadrático** | Memoria y compute escalan como `O(T²)` con longitud de secuencia. Limita contexto largo. |
| **Data Loading** | Múltiples workers, prefetching, pin_memory para evitar que CPU sea cuello de botella. |
| **torch.compile** | Compila modelo a código optimizado. Fusiona operaciones. 1.5-2× speedup típico. |
| **Flash Attention** | Attention en bloques sin materializar matriz completa. `O(T)` memoria vs `O(T²)`. 2-3× más rápido. |
| **Mixed Precision (AMP)** | FP16 para forward/backward, FP32 para optimizer. 2× más rápido y 2× menos memoria. |
| **Gradient Accumulation** | Acumula gradientes sobre múltiples micro-batches. Permite batch size efectivo mayor sin más memoria. |
| **DDP (Distributed Data Parallel)** | Entrenamiento en múltiples GPUs. Escalado casi lineal. Gradientes sincronizados automáticamente. |

---

**Siguiente notebook →** [14 - LLM Pretraining](./14_llm_pretraining.ipynb): entrenar un LLM desde cero con todas estas optimizaciones.